In [ ]:
import pandas as pd

In [ ]:
data = pd.read_csv("../data/heart_failure_clinical_records_dataset.csv")

## 상관행렬 확인

In [ ]:
correlation_matrix = data.corr()

In [ ]:
correlation_matrix

## 타임 변수 히스토그램

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.hist(data['time'], bins=20, edgecolor='k', alpha=0.7)
plt.title('Distribution of Time Variable')
plt.xlabel('Time (days)')
plt.ylabel('Frequency')
plt.grid(True)
plt.show()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import matthews_corrcoef

## 모델 적용

### 데이터 정리

In [ ]:
X = data.drop(columns=['time', 'DEATH_EVENT'])
y = data['DEATH_EVENT']

scaler = StandardScaler()
continuous_vars = ['age', 'creatinine_phosphokinase', 'ejection_fraction', 
                   'platelets', 'serum_creatinine', 'serum_sodium']

X[continuous_vars] = scaler.fit_transform(X[continuous_vars])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=11, stratify=y)

In [ ]:
log_reg = LogisticRegression(random_state=11)
log_reg.fit(X_train, y_train)
y_pred_log_reg = log_reg.predict(X_test)
log_reg_acc = accuracy_score(y_test, y_pred_log_reg)
log_reg_cm = confusion_matrix(y_test, y_pred_log_reg)

In [ ]:
skf = StratifiedKFold(n_splits=5)
log_reg_strat_cv = cross_val_score(log_reg, X_train, y_train, cv=skf, scoring='accuracy')
log_reg_strat_acc = log_reg_strat_cv.mean()

In [ ]:
rf_clf = RandomForestClassifier(random_state=11)
rf_clf.fit(X_train, y_train)
y_pred_rf = rf_clf.predict(X_test)
rf_acc = accuracy_score(y_test, y_pred_rf)
rf_cm = confusion_matrix(y_test, y_pred_rf)

In [ ]:
results = {
    'Model': ['Logistic Regression', 'Stratified Logistic Regression (CV)', 'Random Forest'],
    'Accuracy': [log_reg_acc, log_reg_strat_acc, rf_acc],
    'Confusion Matrix': [log_reg_cm, None, rf_cm]
}

log_reg_mcc = matthews_corrcoef(y_test, y_pred_log_reg)
rf_mcc = matthews_corrcoef(y_test, y_pred_rf)

results['MCC'] = [log_reg_mcc, None, rf_mcc]

In [ ]:
results

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import matthews_corrcoef
import numpy as np

# StratifiedKFold 설정
skf = StratifiedKFold(n_splits=5)

# MCC 값을 저장할 리스트
mcc_scores = []

# 각 폴드에서 MCC 계산
for train_index, test_index in skf.split(X_train, y_train):
    X_train_fold, X_test_fold = X_train[train_index], X_train[test_index]
    y_train_fold, y_test_fold = y_train.iloc[train_index], y_train.iloc[test_index]
    
    # 로지스틱 회귀 모델 훈련
    log_reg.fit(X_train_fold, y_train_fold)
    
    # 예측 및 MCC 계산
    y_pred_fold = log_reg.predict(X_test_fold)
    mcc_fold = matthews_corrcoef(y_test_fold, y_pred_fold)
    
    # 폴드별 MCC 저장
    mcc_scores.append(mcc_fold)

# 폴드별 MCC 평균
mean_mcc = np.mean(mcc_scores)

print(f"Stratified Logistic Regression MCC (Cross-Validation): {mean_mcc}")


## 생존 분석

In [ ]:
!pip install lifelines
from lifelines import CoxPHFitter, KaplanMeierFitter

In [ ]:
# Kaplan-Meier 생존 곡선
kmf = KaplanMeierFitter()
kmf.fit(data['time'], event_observed=data['DEATH_EVENT'])
kmf.plot_survival_function()
plt.title('Kaplan-Meier Survival Curve')
plt.xlabel('Time (days)')
plt.ylabel('Survival Probability')
plt.show()

# Cox Proportional Hazards Model 적용
cph = CoxPHFitter()
cph.fit(data, duration_col='time', event_col='DEATH_EVENT')
cph.print_summary()

# 변수들의 위험 비율 확인
cph.plot()
plt.show()